# Tratamento e Limpeza de Dados

## 1. Regras por Coluna

> **Nota:** Todas as regras de descarte ou imputação abaixo devem manter um **contador** do número de ocorrências registadas.

### Identificação e Âmbito
* **`id`**: Deve ser único. Registos sem `id` ou duplicados (a partir da 2.ª ocorrência) são **descartados**. *(Com contador)*
* **`operation`**: Irrelevante para o estudo final, mas utilizada como filtro — apenas registos de compra (`comprar` / `sale`) são mantidos; os restantes são **descartados**. *(Com contador)*

### Variáveis Numéricas Obrigatórias
* **`price_eur`**: Não pode ser `< 0`, vazio, `NaN` ou não-numérico. Caso contrário, o registo é **descartado**. *(Com contador)*
* **`area_m2`**: Não pode ser `< 0`, vazio, `NaN` ou não-numérico. Caso contrário, o registo é **descartado**. *(Com contador)*
* **`bathrooms`**: Não pode ser `< 0`, vazio, `NaN` ou não-numérico. Caso contrário, o registo é **descartado**. *(Com contador)*

### Geolocalização (Distrito do Porto)
* **`latitude` e `longitude`**:
  * Devem ser valores numéricos válidos e situar-se dentro dos **limites geográficos do distrito do Porto** (latitude em $[40.75, 41.50]$ e longitude em $[-8.80, -8.30]$).
  * **Não** devem ser preenchidas com recurso à coluna `municipality`.
  * Se estiverem vazias, `NaN`, não-numéricas ou fora dos limites do distrito, deve ser feito **`print` do `id`** correspondente e o registo é **descartado**. *(Com contador)*

### Características e Tipologia do Imóvel
* **`property_type`**: Deve ser uma *string* igual a `'house'` ou `'apartment'`. Valores vazios, `NaN`, não-*string* ou diferentes destas categorias são **descartados**. *(Com contador)*
* **`bedrooms`**:
  * Se for `< 0`, o registo é **descartado**.
  * Se for vazio, `NaN` ou não-numérico, preenche-se extraindo o valor numérico da coluna `typology` (removendo o `'T'`). Se continuar inválido, o registo é **descartado**. *(Com contador)*
* **`floor`**: Se for `< 0`, vazio, `NaN` ou não-numérico, assume **`0`** se `property_type == 'house'` ou **`1`** se `property_type == 'apartment'`. *(Com contador)*
* **`num_floors`**: Se for `< 0`, vazio, `NaN` ou não-numérico, assume **`1`** (para `house` ou `apartment`). *(Com contador)*

### Variáveis Booleanas
* **`has_garage`**: Deve ser booleano. Se for vazio, `NaN` ou não-booleano, assume **`False`**. Posteriormente é convertido para inteiro (`1` / `0`). *(Com contador)*
* **`has_garden`**: Deve ser booleano. Se for vazio, `NaN` ou não-booleano, assume **`False`**. Posteriormente é convertido para inteiro (`1` / `0`). *(Com contador)*

---

## 2. Seleção de Colunas

### Colunas Irrelevantes (Removidas após o tratamento)
* `municipality` *(não utilizada para imputação)*
* `operation` *(utilizada apenas como filtro)*
* `typology` *(utilizada apenas para imputar `bedrooms`)*
* `url`, `title`, `location`, `scraped_at`, `matched_address`, `geocode_status`

### Colunas Úteis Mantidas no Dataset Final
* **Alvo (Target):** `price_eur`
* **Numéricas:** `area_m2`, `latitude`, `longitude`, `bedrooms`, `bathrooms`, `floor`, `num_floors`
* **Categóricas / Binárias:** `property_type`, `has_garage`, `has_garden`

---

## 3. Exportação e Validação
* Garantir que não existem valores `NaN` no *dataset* final.
* Apresentar o relatório com todos os contadores de tratamento.
* Guardar o *dataset* limpo em formato **CSV** (`dataset_trabalhado.csv`).

In [ ]:
#### 1. Tratamento de Dados
import numpy as np
import pandas as pd

# Carregar dataset original
df = pd.read_csv('../data/porto_district_all_geocoded.csv', sep=',', on_bad_lines='skip')
df_clean = df.copy()

# Dicionário para guardar todos os contadores exigidos
contadores = {}
linhas_iniciais = len(df_clean)

# Padronizar strings vazias ou apenas com espaços para np.nan
df_clean = df_clean.replace(r"^\s*$", np.nan, regex=True)

In [ ]:
# ---------------------------------------------------------
# 1. ID ÚNICO (descarta nulos/vazios e a 2ª+ ocorrência)
# ---------------------------------------------------------
if "id" in df_clean.columns:
    mask_id_invalido = df_clean["id"].isna() | df_clean["id"].duplicated(keep="first")
else:
    mask_id_invalido = df_clean.index.isna() | df_clean.index.duplicated(keep="first")

contadores["id_invalido_ou_duplicado_descartados"] = int(mask_id_invalido.sum())
df_clean = df_clean[~mask_id_invalido].copy()

In [ ]:
# ---------------------------------------------------------
# 2. OPERATION (garantir que é comprar / sale)
# ---------------------------------------------------------
op_norm = df_clean["operation"].astype(str).str.strip().str.lower()
mask_op_valida = df_clean["operation"].notna() & op_norm.isin(["comprar", "sale"])
contadores["operation_nao_comprar_descartados"] = int((~mask_op_valida).sum())
df_clean = df_clean[mask_op_valida].copy()

In [ ]:
# ---------------------------------------------------------
# 3. PRICE_EUR (não pode ser < 0, vazio, NaN ou não-número)
# ---------------------------------------------------------
price_num = pd.to_numeric(df_clean["price_eur"], errors="coerce")
mask_price_valido = price_num.notna() & (price_num >= 0)
contadores["price_eur_invalido_descartados"] = int((~mask_price_valido).sum())
df_clean = df_clean[mask_price_valido].copy()
df_clean["price_eur"] = price_num[mask_price_valido]

In [ ]:
# ---------------------------------------------------------
# 4. AREA_M2 (não pode ser < 0, vazio, NaN ou não-número)
# ---------------------------------------------------------
area_num = pd.to_numeric(df_clean["area_m2"], errors="coerce")
mask_area_valida = area_num.notna() & (area_num >= 0)
contadores["area_m2_invalido_descartados"] = int((~mask_area_valida).sum())
df_clean = df_clean[mask_area_valida].copy()
df_clean["area_m2"] = area_num[mask_area_valida]

In [ ]:
# ---------------------------------------------------------
# 5. PROPERTY_TYPE (tem de ser string 'house' ou 'apartment')
# ---------------------------------------------------------
eh_string = df_clean["property_type"].apply(lambda x: isinstance(x, str))
prop_norm = df_clean["property_type"].astype(str).str.strip().str.lower()
mask_prop_valida = eh_string & prop_norm.isin(["house", "apartment"])
contadores["property_type_invalido_descartados"] = int((~mask_prop_valida).sum())
df_clean = df_clean[mask_prop_valida].copy()
df_clean["property_type"] = prop_norm[mask_prop_valida]

In [ ]:
# ---------------------------------------------------------
# 6. BEDROOMS (< 0 descarta; vazio/NaN/não-número preenche com typology sem 'T', senão descarta)
# ---------------------------------------------------------
bed_num = pd.to_numeric(df_clean["bedrooms"], errors="coerce")

# 6a. Descartar imediatamente se for número menor que 0
mask_bed_negativo = bed_num.notna() & (bed_num < 0)
contadores["bedrooms_menor_que_0_descartados"] = int(mask_bed_negativo.sum())
df_clean = df_clean[~mask_bed_negativo].copy()
bed_num = bed_num[~mask_bed_negativo]

# 6b. Preencher vazios/NaN/não-números usando typology sem 'T'
mask_bed_ausente = bed_num.isna()
typ_sem_t = pd.to_numeric(
    df_clean.loc[mask_bed_ausente, "typology"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.lstrip("T"),
    errors="coerce",
)
typ_sem_t = typ_sem_t.where(typ_sem_t >= 0, np.nan)
bed_num.loc[mask_bed_ausente] = typ_sem_t

contadores["bedrooms_preenchidos_com_typology"] = int(typ_sem_t.notna().sum())

# 6c. Descartar linhas que continuam sem bedrooms válido após tentar typology
mask_bed_final_invalido = bed_num.isna()
contadores["bedrooms_sem_valor_nem_typology_descartados"] = int(mask_bed_final_invalido.sum())
contadores["bedrooms_total_ocorrencias"] = (
    contadores["bedrooms_menor_que_0_descartados"]
    + contadores["bedrooms_preenchidos_com_typology"]
    + contadores["bedrooms_sem_valor_nem_typology_descartados"]
)

df_clean = df_clean[~mask_bed_final_invalido].copy()
df_clean["bedrooms"] = bed_num[~mask_bed_final_invalido]

In [ ]:
# ---------------------------------------------------------
# 7. BATHROOMS (não pode ser < 0, vazio, NaN ou não-número)
# ---------------------------------------------------------
bath_num = pd.to_numeric(df_clean["bathrooms"], errors="coerce")
mask_bath_valido = bath_num.notna() & (bath_num >= 0)
contadores["bathrooms_invalido_descartados"] = int((~mask_bath_valido).sum())
df_clean = df_clean[mask_bath_valido].copy()
df_clean["bathrooms"] = bath_num[mask_bath_valido]

In [ ]:
# ---------------------------------------------------------
# 8. LATITUDE (vazio/NaN/não-número ou fora de [40.75, 41.50] -> print ID e descarta)
# ---------------------------------------------------------
lat_num = pd.to_numeric(df_clean["latitude"], errors="coerce")

mask_lat_ausente = lat_num.isna()
mask_lat_fora_limites = lat_num.notna() & ((lat_num < 40.75) | (lat_num > 41.50))
mask_lat_invalida = mask_lat_ausente | mask_lat_fora_limites

contadores["latitude_ausente_descartados"] = int(mask_lat_ausente.sum())
contadores["latitude_fora_limites_descartados"] = int(mask_lat_fora_limites.sum())
contadores["latitude_total_descartados"] = int(mask_lat_invalida.sum())

ids_lat_invalidos = (
    df_clean.loc[mask_lat_invalida, "id"].tolist()
    if "id" in df_clean.columns
    else df_clean.index[mask_lat_invalida].tolist()
)
if ids_lat_invalidos:
    print(
        f"IDs descartados por latitude ausente ou fora de [40.75, 41.50] "
        f"({len(ids_lat_invalidos)}): {ids_lat_invalidos}"
    )

df_clean = df_clean[~mask_lat_invalida].copy()
df_clean["latitude"] = lat_num[~mask_lat_invalida]

In [ ]:
# ---------------------------------------------------------
# 9. LONGITUDE (vazio/NaN/não-número ou fora de [-8.80, -8.30] -> print ID e descarta)
# ---------------------------------------------------------
lon_num = pd.to_numeric(df_clean["longitude"], errors="coerce")

mask_lon_ausente = lon_num.isna()
mask_lon_fora_limites = lon_num.notna() & ((lon_num < -8.80) | (lon_num > -8.30))
mask_lon_invalida = mask_lon_ausente | mask_lon_fora_limites

contadores["longitude_ausente_descartados"] = int(mask_lon_ausente.sum())
contadores["longitude_fora_limites_descartados"] = int(mask_lon_fora_limites.sum())
contadores["longitude_total_descartados"] = int(mask_lon_invalida.sum())

ids_lon_invalidos = (
    df_clean.loc[mask_lon_invalida, "id"].tolist()
    if "id" in df_clean.columns
    else df_clean.index[mask_lon_invalida].tolist()
)
if ids_lon_invalidos:
    print(
        f"IDs descartados por longitude ausente ou fora de [-8.80, -8.30] "
        f"({len(ids_lon_invalidos)}): {ids_lon_invalidos}"
    )

df_clean = df_clean[~mask_lon_invalida].copy()
df_clean["longitude"] = lon_num[~mask_lon_invalida]

In [ ]:
# ---------------------------------------------------------
# 10. FLOOR (< 0, vazio, NaN ou não-número -> 0 se house, 1 se apartment)
# ---------------------------------------------------------
floor_num = pd.to_numeric(df_clean["floor"], errors="coerce")
mask_floor_invalido = floor_num.isna() | (floor_num < 0)
contadores["floor_corrigidos"] = int(mask_floor_invalido.sum())

floor_num[mask_floor_invalido & (df_clean["property_type"] == "house")] = 0
floor_num[mask_floor_invalido & (df_clean["property_type"] == "apartment")] = 1
df_clean["floor"] = floor_num

In [ ]:
# ---------------------------------------------------------
# 11. NUM_FLOORS (< 0, vazio, NaN ou não-número -> 1 se house ou apartment)
# ---------------------------------------------------------
num_floors_num = pd.to_numeric(df_clean["num_floors"], errors="coerce")
mask_num_floors_invalido = num_floors_num.isna() | (num_floors_num < 0)
contadores["num_floors_corrigidos"] = int(mask_num_floors_invalido.sum())

num_floors_num[mask_num_floors_invalido] = 1
df_clean["num_floors"] = num_floors_num

In [ ]:
# ---------------------------------------------------------
# 12 & 13. HAS_GARAGE e HAS_GARDEN (booleano válido, senão False; converter para 1/0)
# ---------------------------------------------------------
mapa_booleanos_validos = {
    True: True,
    False: False,
    1: True,
    0: False,
    1.0: True,
    0.0: False,
    "true": True,
    "false": False,
    "1": True,
    "0": False,
    "1.0": True,
    "0.0": False,
}

for col_bool in ["has_garage", "has_garden"]:
    serie_norm = df_clean[col_bool].apply(
        lambda x: x.strip().lower() if isinstance(x, str) else x
    )
    eh_booleano_valido = serie_norm.isin(mapa_booleanos_validos.keys()) & (
        df_clean[col_bool].notna()
    )
    contadores[f"{col_bool}_invalido_colocado_false"] = int((~eh_booleano_valido).sum())
    df_clean[col_bool] = (
        serie_norm.map(mapa_booleanos_validos).astype("boolean").fillna(False).astype(int)
    )

In [ ]:
# ---------------------------------------------------------
# 14. REMOVER COLUNAS IRRELEVANTES E CODIFICAR VARIÁVEIS
# ---------------------------------------------------------
colunas_irrelevantes = [
    "municipality",
    "operation",
    "url",
    "title",
    "location",
    "typology",
    "scraped_at",
    "matched_address",
    "geocode_status",
]
df_clean = df_clean.drop(columns=[c for c in colunas_irrelevantes if c in df_clean.columns])
df_clean = pd.get_dummies(df_clean, columns=["property_type"], dtype=int)

In [ ]:
# ---------------------------------------------------------
# 15. VALIDAÇÃO FINAL, EXPORTAÇÃO E RELATÓRIO DE CONTADORES
# ---------------------------------------------------------
assert df_clean.isna().sum().sum() == 0, "Erro: Ainda existem valores NaN!"

df_clean.to_csv("../data/dataset_trabalhado.csv", index="id" not in df_clean.columns)

print(f"\nLinhas iniciais: {linhas_iniciais}")
print(f"Linhas finais válidas (df_clean): {len(df_clean)}")
print(f"Total de linhas descartadas: {linhas_iniciais - len(df_clean)}")
print("Ficheiro exportado: dataset_trabalhado.csv\n")

print("=== CONTADORES DE TRATAMENTO ===")
for regra, total in contadores.items():
    print(f"- {regra}: {total}")

print("\nColunas finais:", df_clean.columns.tolist())